# Stage 1 - Fact extraction from BanglaCHQ-Summ summaries

Decomposes each gold summary into atomic, typed facts. These facts are the targets that the deletion (`b1_deletion.py`) and alteration (`b2_alteration.py`) pipelines act on in the **source** document.

Facts are extracted from the **summary**, not the question. We need to remove from the source exactly what the summary claims; extracting from the source instead would target things the summary never mentions.

Output mirrors `dataset/NER/test_tag_columns.csv`: one column per category, facts within a column joined by ` | `.

In [1]:
import os
import sys
import time
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath("source_corruption"))
from common import EXTRACT_FACTS_PROMPT, FACT_CATEGORIES, parse_facts

load_dotenv(os.path.abspath("../../.env"))
api_keys = [k for k in (os.getenv("OPENROUTER_API_KEY"), os.getenv("OPENROUTER_API_KEY_NEW")) if k]
if not api_keys:
    raise ValueError("No API key found! Check the .env file at the repository root.")

MODEL = "openai/gpt-5.6-luna"
SAMPLE_SIZE = 50
SPLIT = "test"

print(f"{len(api_keys)} key(s) loaded. Extraction model: {MODEL}")

2 key(s) loaded. Extraction model: openai/gpt-5.6-luna


In [2]:
INPUT_FILE = f"../../BanglaCHQ-Summ/Dataset/{SPLIT}.csv"
OUTPUT_FACTS = f"facts_{SPLIT}_first{SAMPLE_SIZE}_long.csv"
OUTPUT_TAGS = f"facts_{SPLIT}_first{SAMPLE_SIZE}_tag_columns.csv"

df_raw = pd.read_csv(INPUT_FILE)
print(f"Loaded {INPUT_FILE}: {df_raw.shape}, columns {list(df_raw.columns)}")

# 'question' is the source document, 'summary' is the human-written gold summary.
df = df_raw.head(SAMPLE_SIZE).copy().reset_index(drop=True)
df["question"] = df["question"].astype(str).str.strip()
df["summary"] = df["summary"].astype(str).str.strip()

print(f"\nTaking the first {len(df)} rows.")
print(f"Source length  : mean {df['question'].str.split().str.len().mean():.0f} words")
print(f"Summary length : mean {df['summary'].str.split().str.len().mean():.0f} words")

Loaded ../../BanglaCHQ-Summ/Dataset/test.csv: (235, 4), columns ['id', 'question', 'indices', 'summary']

Taking the first 50 rows.
Source length  : mean 72 words
Summary length : mean 30 words


In [3]:
active_key_index = 0


def call_model(prompt, model_name=MODEL, max_retries=3):
    """Send one prompt to OpenRouter. Rotates keys when one is out of credit."""
    global active_key_index

    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0,
    }

    for attempt in range(max_retries):
        for _ in range(len(api_keys)):
            headers = {
                "Authorization": f"Bearer {api_keys[active_key_index]}",
                "Content-Type": "application/json",
            }
            try:
                response = requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers=headers,
                    json=payload,
                    timeout=120,
                )
            except requests.RequestException as exc:
                print(f"  network error: {exc}")
                break

            if response.status_code == 200:
                return response.json()["choices"][0]["message"]["content"]
            if response.status_code in (401, 402, 403, 429):  # 403 = key limit exceeded
                active_key_index = (active_key_index + 1) % len(api_keys)
                continue
            print(f"  HTTP {response.status_code}: {response.text[:200]}")
            break
        time.sleep(2 * (attempt + 1))

    return None


print("Ready.")

Ready.


In [4]:
records = []
failures = []

for position, row in df.iterrows():
    prompt = EXTRACT_FACTS_PROMPT.format(summary=row["summary"])
    raw = call_model(prompt)

    if raw is None:
        failures.append((row["id"], "no response"))
        print(f"[{position + 1}/{len(df)}] id={row['id']}  FAILED")
        continue

    facts = parse_facts(raw)
    if not facts:
        failures.append((row["id"], "unparseable"))
        print(f"[{position + 1}/{len(df)}] id={row['id']}  UNPARSEABLE")
        continue

    for index, category, text in facts:
        if category not in FACT_CATEGORIES:
            category = "Other"
        records.append(
            {
                "id": row["id"],
                "fact_index": index,
                "category": category,
                "fact": text,
                "source": row["question"],
                "summary": row["summary"],
            }
        )

    print(f"[{position + 1}/{len(df)}] id={row['id']}  {len(facts)} facts")

facts_long = pd.DataFrame(records)
facts_long.to_csv(OUTPUT_FACTS, index=False, encoding="utf-8-sig")
print(f"\nSaved {len(facts_long)} facts to {OUTPUT_FACTS}")
print(f"Failures: {len(failures)}  {failures if failures else ''}")

[1/50] id=10696  7 facts


[2/50] id=16760  7 facts


[3/50] id=1483  10 facts


[4/50] id=22486  3 facts


[5/50] id=20110  8 facts


...

[46/50] id=7476  2 facts


[47/50] id=17727  12 facts


[48/50] id=24392  3 facts


[49/50] id=21042  3 facts


[50/50] id=4376  5 facts

Saved 275 facts to facts_test_first50_long.csv
Failures: 0  


In [5]:
# Pivot to the tag-column layout used by dataset/NER/test_tag_columns.csv:
# one column per category, multiple facts joined by " | ".
pivot = (
    facts_long.sort_values(["id", "fact_index"])
    .groupby(["id", "category"])["fact"]
    .apply(lambda values: " | ".join(values))
    .unstack("category")
)

tag_columns = (
    df[["id", "question", "summary"]]
    .rename(columns={"question": "source"})
    .merge(pivot, on="id", how="left")
)

for category in FACT_CATEGORIES:
    if category not in tag_columns.columns:
        tag_columns[category] = pd.NA

tag_columns = tag_columns[["id", "source", "summary"] + FACT_CATEGORIES]
tag_columns["n_facts"] = tag_columns["id"].map(facts_long.groupby("id").size()).fillna(0).astype(int)

tag_columns.to_csv(OUTPUT_TAGS, index=False, encoding="utf-8-sig")
print(f"Saved {tag_columns.shape} to {OUTPUT_TAGS}")
tag_columns.head(3)

Saved (50, 14) to facts_test_first50_tag_columns.csv


,id,source,summary,Symptom,Health Condition,Medicine,Specialist,Age,Dosage,Medical Procedure,Test Result,Temporal,Other,n_facts
0,10696,আমার বয়স ২৯ । আজ ৪ দিন গলা ব্যথা করছে । গরম ল...,বয়স ২৯ । ৪ দিন হলো গলা ব্যথা । গড়গড়া করে কুলি ...,রোগীর ৪ দিন ধরে গলা ব্যথা রয়েছে। | গড়গড়া কর...,NaN,রোগী প্যারাসিটামল খেয়েছেন। | রোগী অ্যান্টিহিস...,NaN,রোগীর বয়স ২৯ বছর।,NaN,NaN,NaN,NaN,প্যারাসিটামল খেয়েও রোগীর উপকার হয়নি। | অ্যান...,7
1,16760,"মাথার যন্তনা , দুইমাস যাবত ঠিকমত ঘুম হয় না , এ...",২ মাস হলো মাথায় যন্ত্রনা করে । ঘুম হয় না । ঘাড়...,রোগীর ঘুম হয় না। | রোগীর ঘাড়ে ব্যথা হয়। | র...,NaN,নাপা এক্সট্রা খেলে রোগীর ঘুম হয়।,NaN,NaN,NaN,NaN,NaN,রোগীর ২ মাস ধরে মাথায় যন্ত্রণা হচ্ছে।,NaN,7
2,1483,আমার বয়স ২৩ বছর আমার রানের দুই চিপায় চুলকানি হ...,বয়স ২৩ । রানের দুই চিপায় চুলকায় ও কালো হয়ে গেছ...,রোগীর উভয় কুঁচকিতে চুলকানি হয়। | রোগীর উভয় কুঁ...,রোগীর রক্তে অ্যালার্জি আছে।,"রোগী ফ্লুকনাজল, আইসোক্লোক্স, ট্রইজিন এবং ইকনাজ...",NaN,রোগীর বয়স ২৩ বছর।,ডাক্তার ৫টি ফ্লুকনাজল দিয়েছেন।,NaN,NaN,১৫ দিন ওষুধ ব্যবহারের পর উপসর্গ কমেছে।,NaN,10


In [6]:
# Distribution checks. Category skew here decides whether the target draw in
# build_jobs needs stratifying before the run scales up.
print("Facts per summary:")
print(tag_columns["n_facts"].describe().to_string())

print("\nFacts by category:")
counts = facts_long["category"].value_counts()
for category in FACT_CATEGORIES:
    n = int(counts.get(category, 0))
    share = 100 * n / len(facts_long) if len(facts_long) else 0
    print(f"  {category:20s} {n:5d}  {share:5.1f}%")

print("\nDocuments with no alterable fact (would yield no alteration job):")
alterable = {c for c in FACT_CATEGORIES if c != "Other"}
per_doc = facts_long.groupby("id")["category"].apply(lambda s: bool(set(s) & alterable))
print(f"  {int((~per_doc).sum())} of {len(per_doc)}")

Facts per summary:
count    50.00000
mean      5.50000
std       2.78663
min       2.00000
25%       3.00000
50%       5.00000
75%       7.00000
max      12.00000

Facts by category:
  Symptom                129   46.9%
  Health Condition        20    7.3%
  Medicine                30   10.9%
  Specialist               1    0.4%
  Age                     24    8.7%
  Dosage                  13    4.7%
  Medical Procedure        5    1.8%
  Test Result             11    4.0%
  Temporal                33   12.0%
  Other                    9    3.3%

Documents with no alterable fact (would yield no alteration job):
  0 of 50
